In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
!ls "/content/drive/MyDrive/CS543/group project"

CS543_Reproducing_and_Analyzing_SimMIM-main.zip  simmim_multi_epoch_output
mask_ratio_multi_epoch_experiments.ipynb	 smoke_test_output


In [22]:
!unzip -q "/content/drive/MyDrive/CS543/group project/CS543_Reproducing_and_Analyzing_SimMIM-main.zip" -d /content/
!ls /content/

replace /content/CS543_Reproducing_and_Analyzing_SimMIM-main/Project Proposal.docx? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/CS543_Reproducing_and_Analyzing_SimMIM-main/README.md? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/CS543_Reproducing_and_Analyzing_SimMIM-main/external_code_patches/main_simmim.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/CS543_Reproducing_and_Analyzing_SimMIM-main/mask_ratio_experiments (2).ipynb? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/CS543_Reproducing_and_Analyzing_SimMIM-main/mask_ratio_experiments (3).ipynb? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/CS543_Reproducing_and_Analyzing_SimMIM-main/notebooks/Visualization.ipynb? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/CS543_Reproducing_and_Analyzing_SimMIM-main/notebooks/mask_ratio_experiments.ipynb? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/CS543_Reproducing_and_Analyzing_SimMIM-main/notebooks/simmim_smok

In [23]:
%cd /content/CS543_Reproducing_and_Analyzing_SimMIM-main
!ls

/content/CS543_Reproducing_and_Analyzing_SimMIM-main
 external_code_patches		     notebooks
'mask_ratio_experiments (2).ipynb'  'Project Proposal.docx'
'mask_ratio_experiments (3).ipynb'   README.md


In [24]:
%cd /content
!git clone https://github.com/microsoft/SimMIM.git
%cd /content/SimMIM
!ls

/content
fatal: destination path 'SimMIM' already exists and is not an empty directory.
/content/SimMIM
CODE_OF_CONDUCT.md  LICENSE	      models		SECURITY.md
config.py	    logger.py	      optimizer.py	SUPPORT.md
configs		    lr_scheduler.py   __pycache__	utils.py
data		    main_finetune.py  README.md
figures		    main_simmim.py    requirements.txt


In [25]:
!pip install -q timm==0.4.12 yacs termcolor scipy pyyaml

In [26]:
import torch
print(torch.cuda.is_available())
!nvidia-smi

True
Tue May  5 01:39:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+

In [27]:
!cp /content/CS543_Reproducing_and_Analyzing_SimMIM-main/external_code_patches/main_simmim.py /content/SimMIM/main_simmim.py

In [28]:
!ls /content/SimMIM/main_simmim.py

/content/SimMIM/main_simmim.py


In [29]:
from PIL import Image
from pathlib import Path
import numpy as np

dummy_root = Path("/content/dummy_data/train/class0")
dummy_root.mkdir(parents=True, exist_ok=True)

for i in range(3):
    img = Image.fromarray(np.random.randint(0, 255, (192, 192, 3), dtype=np.uint8))
    img.save(dummy_root / f"img_{i}.png")

print("dummy data ready:", dummy_root)

dummy data ready: /content/dummy_data/train/class0


In [30]:
%cd /content/SimMIM

!python -m torch.distributed.launch --nproc_per_node 1 main_simmim.py \
--cfg configs/swin_base__100ep/simmim_pretrain__swin_base__img192_window6__100ep.yaml \
--data-path /content/dummy_data/train \
--batch-size 2 \
--amp-opt-level O0 \
--output "/content/drive/MyDrive/CS543/group project/smoke_test_output" \
--tag smoke_test \
--opts TRAIN.EPOCHS 1 TRAIN.WARMUP_EPOCHS 0 PRINT_FREQ 1 SAVE_FREQ 1 DATA.NUM_WORKERS 2 DATA.MASK_RATIO 0.5

/content/SimMIM
/usr/local/lib/python3.12/dist-packages/torch/distributed/launch.py:207: FutureWarning: The module torch.distributed.launch is deprecated
and will be removed in future. Use torchrun.
Note that --use-env is set by default in torchrun.
If your script expects `--local-rank` argument to be set, please
change it to read from `os.environ['LOCAL_RANK']` instead. See 
https://pytorch.org/docs/stable/distributed.html#launch-utility for 
further instructions

  main()
=> merge config from configs/swin_base__100ep/simmim_pretrain__swin_base__img192_window6__100ep.yaml
RANK and WORLD_SIZE in environ: 0/1
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W505 01:40:01.963986726 ProcessGroupNCCL.cpp:5138] Guessing device ID based on global rank. This can cause a hang if rank to GPU 

In [31]:
!find "/content/drive/MyDrive/CS543/group project/smoke_test_output" -name "*.pth"

/content/drive/MyDrive/CS543/group project/smoke_test_output/simmim_pretrain/smoke_test/ckpt_epoch_0.pth


In [32]:
from torchvision.datasets import CIFAR10
from pathlib import Path

root = Path("/content/cifar10_simmim_subset/train")
root.mkdir(parents=True, exist_ok=True)

dataset = CIFAR10(root="/content/cifar10_data", train=True, download=True)

max_per_class = 200
class_counts = {i: 0 for i in range(10)}

for img, label in dataset:
    if class_counts[label] >= max_per_class:
        continue

    class_dir = root / str(label)
    class_dir.mkdir(parents=True, exist_ok=True)

    img.save(class_dir / f"{class_counts[label]:04d}.png")
    class_counts[label] += 1

    if all(v >= max_per_class for v in class_counts.values()):
        break

print(class_counts)
print("Saved to:", root)

{0: 200, 1: 200, 2: 200, 3: 200, 4: 200, 5: 200, 6: 200, 7: 200, 8: 200, 9: 200}
Saved to: /content/cifar10_simmim_subset/train


In [33]:
!find /content/cifar10_simmim_subset/train -type f | wc -l

2000


In [39]:
import subprocess
import re
import time
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

%cd /content/SimMIM

OUTPUT_DIR = Path("/content/drive/MyDrive/CS543/group project/simmim_multi_epoch_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOG_DIR = OUTPUT_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

BASE_CMD = """
python -m torch.distributed.launch --nproc_per_node 1 main_simmim.py \
--cfg configs/swin_base__100ep/simmim_pretrain__swin_base__img192_window6__100ep.yaml \
--data-path /content/cifar10_simmim_subset/train \
--batch-size 2 \
--amp-opt-level O0 \
--output "{output_dir}" \
--tag {tag} \
--opts TRAIN.EPOCHS {epochs} TRAIN.WARMUP_EPOCHS 0 PRINT_FREQ 1 SAVE_FREQ 1 DATA.NUM_WORKERS 2 DATA.MASK_RATIO {ratio}
"""

ratios = [0.25, 0.50, 0.75]
epochs_list = [1, 5, 10]

records = []

for ratio in ratios:
    for epochs in epochs_list:
        tag = f"mask{int(ratio*100):03d}_epoch{epochs}"

        cmd = BASE_CMD.format(
            output_dir=str(OUTPUT_DIR),
            tag=tag,
            epochs=epochs,
            ratio=ratio
        )

        print("=" * 100)
        print(f"Running: mask_ratio={ratio}, epochs={epochs}, tag={tag}")
        print("=" * 100)

        start_time = time.time()

        result = subprocess.run(
            cmd,
            shell=True,
            text=True,
            capture_output=True
        )

        end_time = time.time()
        training_time_sec = end_time - start_time

        output = result.stdout + "\n" + result.stderr

        # Save raw wrapper log
        log_path = LOG_DIR / f"{tag}.txt"
        log_path.write_text(output)

        # Combine stdout/stderr with SimMIM internal log if it exists
        combined_output = output
        internal_logs = list(OUTPUT_DIR.rglob(f"*{tag}*/log_rank0.txt"))
        for internal_log in internal_logs:
            try:
                combined_output += "\n" + internal_log.read_text()
            except Exception:
                pass

        # Extract losses more flexibly
        losses = [
            float(x)
            for x in re.findall(r"loss[:\s]+([0-9]*\.?[0-9]+)", combined_output, flags=re.IGNORECASE)
        ]

        final_loss = losses[-1] if len(losses) > 0 else np.nan
        avg_loss = float(np.mean(losses)) if len(losses) > 0 else np.nan

        # Find expected checkpoint
        ckpt_candidates = list(OUTPUT_DIR.rglob(f"*{tag}*/ckpt_epoch_{epochs-1}.pth"))
        checkpoint_path = str(ckpt_candidates[0]) if len(ckpt_candidates) > 0 else "not_found"

        records.append({
            "mask_ratio": ratio,
            "epochs": epochs,
            "final_loss": final_loss,
            "avg_loss": avg_loss,
            "num_loss_values": len(losses),
            "training_time_sec": training_time_sec,
            "training_time_min": training_time_sec / 60,
            "checkpoint": checkpoint_path,
            "log_file": str(log_path),
            "return_code": result.returncode
        })

        print(f"Final loss: {final_loss}")
        print(f"Avg loss: {avg_loss}")
        print(f"Num loss values parsed: {len(losses)}")
        print(f"Training time: {training_time_sec/60:.2f} min")
        print(f"Checkpoint: {checkpoint_path}")
        print(f"Log file: {log_path}")
        print(f"Return code: {result.returncode}")

        if result.returncode != 0:
            print("\nERROR DETECTED. Last 3000 characters of log:")
            print(combined_output[-3000:])
            break

    if len(records) > 0 and records[-1]["return_code"] != 0:
        print("Stopping because one experiment failed.")
        break

results_df = pd.DataFrame(records)
display(results_df)

csv_path = OUTPUT_DIR / "multi_epoch_results.csv"
results_df.to_csv(csv_path, index=False)

print("Saved CSV to:", csv_path)

/content/SimMIM
Running: mask_ratio=0.25, epochs=1, tag=mask025_epoch1
Final loss: 0.592
Avg loss: 0.8061750999999999
Num loss values parsed: 1000
Training time: 0.36 min
Checkpoint: /content/drive/MyDrive/CS543/group project/simmim_multi_epoch_output/simmim_pretrain/mask025_epoch1/ckpt_epoch_0.pth
Log file: /content/drive/MyDrive/CS543/group project/simmim_multi_epoch_output/logs/mask025_epoch1.txt
Return code: 0
Running: mask_ratio=0.25, epochs=5, tag=mask025_epoch5
Final loss: 0.5034
Avg loss: 0.5943813784764207
Num loss values parsed: 9097
Training time: 10.12 min
Checkpoint: /content/drive/MyDrive/CS543/group project/simmim_multi_epoch_output/simmim_pretrain/mask025_epoch5/ckpt_epoch_4.pth
Log file: /content/drive/MyDrive/CS543/group project/simmim_multi_epoch_output/logs/mask025_epoch5.txt
Return code: 0
Running: mask_ratio=0.25, epochs=10, tag=mask025_epoch10
Final loss: 0.4583
Avg loss: 0.5241150099999999
Num loss values parsed: 20000
Training time: 26.01 min
Checkpoint: /conte

,mask_ratio,epochs,final_loss,avg_loss,num_loss_values,training_time_sec,training_time_min,checkpoint,log_file,return_code
0,0.25,1,0.5920,0.806175,1000,21.739612,0.362327,/content/drive/MyDrive/CS543/group project/sim...,/content/drive/MyDrive/CS543/group project/sim...,0
1,0.25,5,0.5034,0.594381,9097,607.227781,10.120463,/content/drive/MyDrive/CS543/group project/sim...,/content/drive/MyDrive/CS543/group project/sim...,0
2,0.25,10,0.4583,0.524115,20000,1560.307635,26.005127,/content/drive/MyDrive/CS543/group project/sim...,/content/drive/MyDrive/CS543/group project/sim...,0
3,0.50,1,0.5943,0.818151,2000,167.622374,2.793706,/content/drive/MyDrive/CS543/group project/sim...,/content/drive/MyDrive/CS543/group project/sim...,0
4,0.50,5,0.6651,0.649077,10000,831.252083,13.854201,/content/drive/MyDrive/CS543/group project/sim...,/content/drive/MyDrive/CS543/group project/sim...,0
5,0.50,10,0.5224,0.572494,20000,1566.102202,26.101703,/content/drive/MyDrive/CS543/group project/sim...,/content/drive/MyDrive/CS543/group project/sim...,0
6,0.75,1,0.6225,0.849049,2000,166.901151,2.781686,/content/drive/MyDrive/CS543/group project/sim...,/content/drive/MyDrive/CS543/group project/sim...,0
7,0.75,5,0.7534,0.725249,10000,772.562207,12.876037,/content/drive/MyDrive/CS543/group project/sim...,/content/drive/MyDrive/CS543/group project/sim...,0
8,0.75,10,0.6614,0.667173,20000,1531.467522,25.524459,/content/drive/MyDrive/CS543/group project/sim...,/content/drive/MyDrive/CS543/group project/sim...,0


Saved CSV to: /content/drive/MyDrive/CS543/group project/simmim_multi_epoch_output/multi_epoch_results.csv


In [41]:
from pathlib import Path
import re
import numpy as np

OUTPUT_DIR = Path("/content/drive/MyDrive/CS543/group project/simmim_multi_epoch_output")
tag = "mask025_epoch1"

wrapper_log = OUTPUT_DIR / "logs" / f"{tag}.txt"
internal_log = OUTPUT_DIR / "simmim_pretrain" / tag / "log_rank0.txt"

combined_output = ""

if wrapper_log.exists():
    combined_output += wrapper_log.read_text()

if internal_log.exists():
    combined_output += "\n" + internal_log.read_text()

losses = [
    float(x)
    for x in re.findall(r"loss[:\s]+([0-9]*\.?[0-9]+)", combined_output, flags=re.IGNORECASE)
]

final_loss = losses[-1] if len(losses) > 0 else np.nan
avg_loss = float(np.mean(losses)) if len(losses) > 0 else np.nan

print("Final loss:", final_loss)
print("Avg loss:", avg_loss)
print("Num loss values parsed:", len(losses))
print("Internal log:", internal_log)

Final loss: 0.592
Avg loss: 0.8061750999999999
Num loss values parsed: 1000
Internal log: /content/drive/MyDrive/CS543/group project/simmim_multi_epoch_output/simmim_pretrain/mask025_epoch1/log_rank0.txt


In [42]:
import pandas as pd
from pathlib import Path

csv_path = Path("/content/drive/MyDrive/CS543/group project/simmim_multi_epoch_output/multi_epoch_results.csv")

df = pd.read_csv(csv_path)

checkpoint_path = str(OUTPUT_DIR / "simmim_pretrain" / tag / "ckpt_epoch_0.pth")
log_path = str(wrapper_log)

new_row = {
    "mask_ratio": 0.25,
    "epochs": 1,
    "final_loss": final_loss,
    "avg_loss": avg_loss,
    "num_loss_values": len(losses),
    "training_time_sec": 33.0,   # 你刚才显示 0.55 min，大约 33 sec
    "training_time_min": 0.55,
    "checkpoint": checkpoint_path,
    "log_file": log_path,
    "return_code": 0
}

df = df[~((df["mask_ratio"] == 0.25) & (df["epochs"] == 1))]
df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
df = df.sort_values(["mask_ratio", "epochs"]).reset_index(drop=True)

df.to_csv(csv_path, index=False)
df

,mask_ratio,epochs,final_loss,avg_loss,num_loss_values,training_time_sec,training_time_min,checkpoint,log_file,return_code
0,0.25,1,0.5920,0.806175,1000,33.000000,0.550000,/content/drive/MyDrive/CS543/group project/sim...,/content/drive/MyDrive/CS543/group project/sim...,0
1,0.25,5,0.5034,0.594381,9097,607.227781,10.120463,/content/drive/MyDrive/CS543/group project/sim...,/content/drive/MyDrive/CS543/group project/sim...,0
2,0.25,10,0.4583,0.524115,20000,1560.307635,26.005127,/content/drive/MyDrive/CS543/group project/sim...,/content/drive/MyDrive/CS543/group project/sim...,0
3,0.50,1,0.5943,0.818151,2000,167.622374,2.793706,/content/drive/MyDrive/CS543/group project/sim...,/content/drive/MyDrive/CS543/group project/sim...,0
4,0.50,5,0.6651,0.649077,10000,831.252083,13.854201,/content/drive/MyDrive/CS543/group project/sim...,/content/drive/MyDrive/CS543/group project/sim...,0
5,0.50,10,0.5224,0.572494,20000,1566.102202,26.101703,/content/drive/MyDrive/CS543/group project/sim...,/content/drive/MyDrive/CS543/group project/sim...,0
6,0.75,1,0.6225,0.849049,2000,166.901151,2.781686,/content/drive/MyDrive/CS543/group project/sim...,/content/drive/MyDrive/CS543/group project/sim...,0
7,0.75,5,0.7534,0.725249,10000,772.562207,12.876037,/content/drive/MyDrive/CS543/group project/sim...,/content/drive/MyDrive/CS543/group project/sim...,0
8,0.75,10,0.6614,0.667173,20000,1531.467522,25.524459,/content/drive/MyDrive/CS543/group project/sim...,/content/drive/MyDrive/CS543/group project/sim...,0


In [ ]:
#upload to github

In [49]:
%cd /content
!git clone https://github.com/alina021001/CS543_Reproducing_and_Analyzing_SimMIM.git github_repo
%cd /content/github_repo
!git status

/content
Cloning into 'github_repo'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 30 (delta 11), reused 7 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 849.70 KiB | 10.36 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/github_repo
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
